# Pipeline Control Notebook
Use these interactive cells to run the Scraping, Transformation, and Ingestion stages manually.


In [ ]:
!pip install -r requirements.txt

In [9]:
import os
import sys
from pathlib import Path

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_ROOT = '/content/drive/Othercomputers/Lenovo L15/Desktop/repo_collabs/ds_projects_collabs/2-nlp-astroturfing-report'
    if PROJECT_ROOT not in sys.path:
        sys.path.append(PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

    # For Colab, manually upload your service account JSON to the ephemeral /content/ folder
    # This secures your key without syncing it to Google Drive!
    try:
        from google.colab import userdata
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = userdata.get('GCP_JSON_PATH')
        os.environ['GCP_PROJECT_ID'] = userdata.get('GCP_PROJECT_ID')
    except Exception:
        print('Please open the Colab Secrets (Key Icon) and add:')
        print('- GCP_PROJECT_ID : your project id')
        print('- GCP_JSON_PATH : /content/your_service_account.json')
        print('Remember to upload the actual .json file physically to the /content/ folder every time you start a new session!')
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = input('Path to uploaded JSON: ')
        os.environ['GCP_PROJECT_ID'] = input('GCP Project ID: ')
else:
    # Running locally in VS Code
    current_dir = Path().absolute()
    project_root = current_dir.parent
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))

# Import modules
from src.forensics.manual_scrapping import scrape_submission_url
from src.infra.data_transformation import transform_to_structured
from src.infra.gcp_ingestion import load_to_bigquery


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Please open the Colab Secrets (Key Icon) and add:
- GCP_PROJECT_ID : your project id
- GCP_JSON_PATH : /content/your_service_account.json
Remember to upload the actual .json file physically to the /content/ folder every time you start a new session!
Path to uploaded JSON: /astroturfing-report-52d14c879cc5.json
GCP Project ID: astroturfing-report


# Phase 1. Scrapping and GCP upload.

## 1. Scrape a Reddit Thread
Replace the URL below with the submission you want to scrape.

In [ ]:
## WARNING this will take time, 1 sec per user scrapped from the URL used. ##
TARGET_URL = "https://www.reddit.com/r/mexico/comments/1rcb63u/i_hope_you_win_your_fight_against_the_cartel/" # Example URL

print(f"Starting manual scrape for: {TARGET_URL}")
raw_file_path = scrape_submission_url(TARGET_URL)

if raw_file_path:
    print(f"Scraping complete. Raw file saved at: {raw_file_path}")
else:
    print("Scraping failed.")


## 2. Transform Phase
Define your mode:
- `master`: Transforms all JSONs into a single master CSV (replaces BigQuery table entirely).
- `single`: Transforms only the latest raw file (appends to BigQuery table).

In [ ]:
MODE = "master"
print(f"Starting transformation in {MODE} mode...")

if MODE == "master":
    print("Consolidating all raw data into master CSV...")
    structured_file = transform_to_structured(
        input_path="data/raw",
        output_dir="data/structured",
        format="csv"
    )
    write_disposition = "WRITE_TRUNCATE"
else:
    print(f"Transforming single submission: {raw_file_path}")
    structured_file = transform_to_structured(
        input_path=raw_file_path,
        output_dir="data/structured",
        format="csv"
    )
    write_disposition = "WRITE_APPEND"

if structured_file:
    print(f"Transformation complete. Structured file ready at: {structured_file}")
else:
    print("Transformation failed.")


## 3. Ingestion Phase
Push the structured data to BigQuery.

In [ ]:
# Set the file to upload. To use an existing file instead of the pipeline output, edit these variables:
try:
    file_to_upload = structured_file
    current_write_disposition = write_disposition
except NameError:
    file_to_upload = r'C:\Users\arq_c\Desktop\repo_collabs\ds_projects_collabs\2-nlp-astroturfing-report\data\structured\transformed_comments.csv'
    current_write_disposition = 'WRITE_APPEND'

print(f"Syncing {file_to_upload} to BigQuery...")
print(f"Table operation mode: {current_write_disposition}")

try:
    load_to_bigquery(
        file_path=file_to_upload,
        table_name="comments_structured",
        write_disposition=current_write_disposition
    )
    print("Ingestion complete. Data is now in GCP.")
except Exception as e:
    print(f"Ingestion failed: {e}")


# Phase 2: Run GCP Transformations (Medallion Architecture)


## 1. BigQuery SQL implementations.
Executes the Bronze -> Silver -> Gold SQL pipelines in BigQuery.

In [ ]:
from src.infra.run_transformations import run_all_transformations

print("Starting GCP SQL Transformations...")
try:
    run_all_transformations()
    print("Transformations completed successfully!")
except Exception as e:
    print(f"Transformation failed: {e}")


2026-03-02 11:55:39.013 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 01_bronze.sql...


Starting GCP SQL Transformations...


2026-03-02 11:55:41.220 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 01_bronze.sql
2026-03-02 11:55:41.221 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 02_silver.sql...
2026-03-02 11:55:43.616 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 02_silver.sql
2026-03-02 11:55:43.617 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 03_gold.sql...
2026-03-02 11:55:46.235 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 03_gold.sql
2026-03-02 11:55:46.236 | INFO     | src.infra.run_transformations:run_sql_file:20 - Executing 04_gold_nlp.sql...
2026-03-02 11:55:48.375 | SUCCESS  | src.infra.run_transformations:run_sql_file:23 - Successfully executed 04_gold_nlp.sql


Transformations completed successfully!


# Phase 3: NLP Processing (Colab Mode)
This phase will pull data from the BigQuery `gold_nlp` table, run heavy BERT Embeddings and Clustering, perform Sentiment Analysis, and push the results back to a new `gold_nlp_results` table.

In [ ]:
# Run this ONCE if executing in Google Colab to install heavy dependencies
import sys
if 'google.colab' in sys.modules:
    !pip install -r requirements.txt
else:
    print('Running locally. Assuming requirements are already installed.')


In [11]:
from src.nlp.nlp_pipeline import run_nlp_pipeline

print("Initializing NLP Pipeline... This may take several minutes on the first run as models download.")
run_nlp_pipeline()


2026-03-02 21:16:46.431 | INFO     | src.nlp.nlp_pipeline:run_nlp_pipeline:7 - === Starting Phase 2: NLP Pipeline ===
2026-03-02 21:16:46.474 | INFO     | src.nlp.data_loader:pull_gold_nlp:15 - Downloading NLP input data from reddit_scrap.gold_nlp...


Initializing NLP Pipeline... This may take several minutes on the first run as models download.


2026-03-02 21:16:47.698 | SUCCESS  | src.nlp.data_loader:pull_gold_nlp:17 - Downloaded 189 records.
2026-03-02 21:16:47.699 | INFO     | src.nlp.embeddings_cluster:generate_clusters:9 - Loading multilingual embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-02 21:16:50.964 | INFO     | src.nlp.embeddings_cluster:generate_clusters:13 - Generating embeddings...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

2026-03-02 21:16:59.321 | INFO     | src.nlp.embeddings_cluster:generate_clusters:16 - Reducing dimensionality with UMAP...
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
2026-03-02 21:16:59.651 | INFO     | src.nlp.embeddings_cluster:generate_clusters:20 - Clustering with HDBSCAN...
2026-03-02 21:16:59.657 | SUCCESS  | src.nlp.embeddings_cluster:generate_clusters:28 - Found 2 clusters (excluding noise -1).
2026-03-02 21:16:59.657 | INFO     | src.nlp.sentiment_analysis:analyze_sentiment:7 - Loading multilingual sentiment model...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-02 21:17:01.102 | INFO     | src.nlp.sentiment_analysis:analyze_sentiment:11 - Running sentiment analysis...
2026-03-02 21:17:49.828 | SUCCESS  | src.nlp.sentiment_analysis:analyze_sentiment:18 - Sentiment analysis complete.
2026-03-02 21:17:50.126 | INFO     | src.nlp.data_loader:push_nlp_results:27 - Uploading NLP results to astroturfing-report.reddit_scrap.gold_nlp_results...
2026-03-02 21:17:53.699 | SUCCESS  | src.nlp.data_loader:push_nlp_results:30 - Upload complete.
